# Assignment 2: Credit Card Fraud Detection Using ANN

## Objective
Use an Artificial Neural Network (ANN) to classify credit-card transactions as:

- **0 = Legitimate / Normal**
- **1 = Fraudulent**

**Dataset:** Credit Card Fraud Dataset  
**CSV File:** `creditcard.csv`  
**Target Column:** `Class`

### Dataset Overview
This notebook uses the uploaded dataset and performs complete preprocessing, class-imbalance handling, ANN experiments, evaluation, prediction, and model comparison.

> **Important:** Accuracy alone is not sufficient for fraud detection. Precision, Recall and F1-score are also considered, with particular attention to **Recall** because detecting fraudulent transactions is important.


In [1]:
# 1. Import Libraries

# If TensorFlow / imbalanced-learn are not installed, run:
# !pip install tensorflow imbalanced-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Input, Dropout
from tensorflow.keras.callbacks import EarlyStopping

print("TensorFlow version:", tf.__version__)


ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
# 2. Load Dataset

file_path = "creditcard.csv"

df = pd.read_csv(file_path)

print("Dataset Shape:", df.shape)
display(df.head())


In [ ]:
# 3. Basic EDA

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nColumn Names:")
print(df.columns.tolist())

print("\nDataset Information:")
df.info()


In [ ]:
# 4. Statistical Summary

display(df.describe().T)


In [ ]:
# 5. Missing Values

missing = df.isnull().sum()

missing_table = pd.DataFrame({
    "Missing Values": missing,
    "Missing Percentage": (missing / len(df) * 100).round(4)
})

display(missing_table[missing_table["Missing Values"] > 0])

print("Total Missing Values:", df.isnull().sum().sum())


In [ ]:
# 6. Duplicate Values

duplicate_count = df.duplicated().sum()

print("Number of duplicate rows:", duplicate_count)

# Remove exact duplicate records before model training
df = df.drop_duplicates().reset_index(drop=True)

print("Shape after removing duplicates:", df.shape)


In [2]:
# 7. Class Distribution

class_counts = df["Class"].value_counts().sort_index()

print("Class Distribution:")
print(class_counts)

print("\nClass Percentage:")
print((df["Class"].value_counts(normalize=True).sort_index() * 100).round(4))

plt.figure(figsize=(7,5))
plt.bar(
    ["Legitimate (0)", "Fraudulent (1)"],
    [class_counts.get(0, 0), class_counts.get(1, 0)]
)
plt.xlabel("Transaction Class")
plt.ylabel("Number of Transactions")
plt.title("Class Distribution")
plt.show()


NameError: name 'df' is not defined

## 8. Feature and Target Separation

`Class` is the target and must not be included in the input features.

The dataset contains `Time`, `Amount`, and anonymized PCA features (`V1`–`V28`). `Class` is excluded from X.


In [3]:
X = df.drop(columns=["Class"])
y = df["Class"]

print("X Shape:", X.shape)
print("y Shape:", y.shape)

print("\nTarget values:")
print(sorted(y.unique()))


NameError: name 'df' is not defined

In [4]:
# 9. Train-Test Split

# Stratify keeps the original fraud/non-fraud proportion in both sets.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)

print("\nTraining class distribution:")
print(y_train.value_counts())

print("\nTesting class distribution:")
print(y_test.value_counts())


NameError: name 'X' is not defined

In [5]:
# 10. Feature Scaling

# Fit the scaler ONLY on training data to avoid data leakage.
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaled Training Shape:", X_train_scaled.shape)
print("Scaled Testing Shape:", X_test_scaled.shape)


NameError: name 'X_train' is not defined

## 11. Handling Class Imbalance Using Class Weights

The fraud class is much smaller than the legitimate class. Instead of artificially balancing the test set, class weights are calculated from the training data only.

This gives greater importance to fraudulent transactions during ANN training while keeping the test data realistic.


In [6]:
classes = np.unique(y_train)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights = dict(zip(classes, weights))

print("Class Weights:")
print(class_weights)


NameError: name 'y_train' is not defined

In [7]:
# 12. ANN Model Function

def create_ann(input_dim, hidden_layers, activation, optimizer, dropout_rate=0.0):
    model = Sequential()
    model.add(Input(shape=(input_dim,)))

    for neurons in hidden_layers:
        model.add(Dense(neurons, activation=activation))
        if dropout_rate > 0:
            model.add(Dropout(dropout_rate))

    # Binary classification output
    model.add(Dense(1, activation="sigmoid"))

    model.compile(
        optimizer=optimizer,
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model


def evaluate_model(model, X_test, y_test):
    probabilities = model.predict(X_test, verbose=0).ravel()
    predictions = (probabilities >= 0.5).astype(int)

    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(y_test, predictions, zero_division=0)
    recall = recall_score(y_test, predictions, zero_division=0)
    f1 = f1_score(y_test, predictions, zero_division=0)
    cm = confusion_matrix(y_test, predictions)

    return predictions, accuracy, precision, recall, f1, cm


early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True
)

print("ANN model function created.")


NameError: name 'EarlyStopping' is not defined

## Experiment 1 — ReLU + Adam

**Architecture:** 64 → 32  
**Activation:** ReLU  
**Optimizer:** Adam  
**Epochs:** 20  
**Batch Size:** 256  
**Class Weights:** Applied


In [8]:
model1 = create_ann(
    input_dim=X_train_scaled.shape[1],
    hidden_layers=[64, 32],
    activation="relu",
    optimizer="adam"
)

history1 = model1.fit(
    X_train_scaled,
    y_train,
    validation_split=0.20,
    epochs=20,
    batch_size=256,
    class_weight=class_weights,
    callbacks=[early_stopping],
    verbose=1
)

pred1, acc1, prec1, rec1, f11, cm1 = evaluate_model(
    model1, X_test_scaled, y_test
)

print("\nModel 1 Results")
print("Accuracy :", round(acc1, 4))
print("Precision:", round(prec1, 4))
print("Recall   :", round(rec1, 4))
print("F1 Score :", round(f11, 4))


NameError: name 'X_train_scaled' is not defined

In [9]:
# Model 1 Loss Graph

plt.figure(figsize=(8,5))
plt.plot(history1.history["loss"], label="Training Loss")
plt.plot(history1.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Binary Cross-Entropy Loss")
plt.title("Model 1 - ReLU + Adam")
plt.legend()
plt.show()


NameError: name 'history1' is not defined

<Figure size 800x500 with 0 Axes>

In [10]:
# Model 1 Confusion Matrix

print("Confusion Matrix - Model 1")
print(cm1)


Confusion Matrix - Model 1


NameError: name 'cm1' is not defined

## Experiment 2 — Tanh + SGD

**Architecture:** 64 → 32  
**Activation:** Tanh  
**Optimizer:** SGD  
**Epochs:** 50  
**Batch Size:** 256  
**Class Weights:** Applied


In [11]:
model2 = create_ann(
    input_dim=X_train_scaled.shape[1],
    hidden_layers=[64, 32],
    activation="tanh",
    optimizer="sgd"
)

history2 = model2.fit(
    X_train_scaled,
    y_train,
    validation_split=0.20,
    epochs=50,
    batch_size=256,
    class_weight=class_weights,
    callbacks=[early_stopping],
    verbose=1
)

pred2, acc2, prec2, rec2, f12, cm2 = evaluate_model(
    model2, X_test_scaled, y_test
)

print("\nModel 2 Results")
print("Accuracy :", round(acc2, 4))
print("Precision:", round(prec2, 4))
print("Recall   :", round(rec2, 4))
print("F1 Score :", round(f12, 4))


NameError: name 'X_train_scaled' is not defined

In [12]:
# Model 2 Loss Graph

plt.figure(figsize=(8,5))
plt.plot(history2.history["loss"], label="Training Loss")
plt.plot(history2.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Binary Cross-Entropy Loss")
plt.title("Model 2 - Tanh + SGD")
plt.legend()
plt.show()


NameError: name 'history2' is not defined

<Figure size 800x500 with 0 Axes>

## Experiment 3 — ReLU + RMSprop

**Architecture:** 128 → 64 → 32  
**Activation:** ReLU  
**Optimizer:** RMSprop  
**Epochs:** 100  
**Batch Size:** 256  
**Class Weights:** Applied


In [13]:
model3 = create_ann(
    input_dim=X_train_scaled.shape[1],
    hidden_layers=[128, 64, 32],
    activation="relu",
    optimizer="rmsprop"
)

history3 = model3.fit(
    X_train_scaled,
    y_train,
    validation_split=0.20,
    epochs=100,
    batch_size=256,
    class_weight=class_weights,
    callbacks=[early_stopping],
    verbose=1
)

pred3, acc3, prec3, rec3, f13, cm3 = evaluate_model(
    model3, X_test_scaled, y_test
)

print("\nModel 3 Results")
print("Accuracy :", round(acc3, 4))
print("Precision:", round(prec3, 4))
print("Recall   :", round(rec3, 4))
print("F1 Score :", round(f13, 4))


NameError: name 'X_train_scaled' is not defined

In [14]:
# Model 3 Loss Graph

plt.figure(figsize=(8,5))
plt.plot(history3.history["loss"], label="Training Loss")
plt.plot(history3.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Binary Cross-Entropy Loss")
plt.title("Model 3 - ReLU + RMSprop")
plt.legend()
plt.show()


NameError: name 'history3' is not defined

<Figure size 800x500 with 0 Axes>

## Experiment 4 — Sigmoid + Adam

**Architecture:** 64 → 32  
**Activation:** Sigmoid  
**Optimizer:** Adam  
**Epochs:** 50  
**Batch Size:** 256  
**Class Weights:** Applied


In [15]:
model4 = create_ann(
    input_dim=X_train_scaled.shape[1],
    hidden_layers=[64, 32],
    activation="sigmoid",
    optimizer="adam"
)

history4 = model4.fit(
    X_train_scaled,
    y_train,
    validation_split=0.20,
    epochs=50,
    batch_size=256,
    class_weight=class_weights,
    callbacks=[early_stopping],
    verbose=1
)

pred4, acc4, prec4, rec4, f14, cm4 = evaluate_model(
    model4, X_test_scaled, y_test
)

print("\nModel 4 Results")
print("Accuracy :", round(acc4, 4))
print("Precision:", round(prec4, 4))
print("Recall   :", round(rec4, 4))
print("F1 Score :", round(f14, 4))


NameError: name 'X_train_scaled' is not defined

In [16]:
# Model 4 Loss Graph

plt.figure(figsize=(8,5))
plt.plot(history4.history["loss"], label="Training Loss")
plt.plot(history4.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Binary Cross-Entropy Loss")
plt.title("Model 4 - Sigmoid + Adam")
plt.legend()
plt.show()


NameError: name 'history4' is not defined

<Figure size 800x500 with 0 Axes>

## Experiment 5 — Deep ReLU + Adam

**Architecture:** 128 → 64 → 32 → 16  
**Activation:** ReLU  
**Optimizer:** Adam  
**Epochs:** 100  
**Batch Size:** 128  
**Dropout:** 0.20  
**Class Weights:** Applied

This experiment changes architecture, batch size, and adds dropout to help control overfitting.


In [17]:
model5 = create_ann(
    input_dim=X_train_scaled.shape[1],
    hidden_layers=[128, 64, 32, 16],
    activation="relu",
    optimizer="adam",
    dropout_rate=0.20
)

history5 = model5.fit(
    X_train_scaled,
    y_train,
    validation_split=0.20,
    epochs=100,
    batch_size=128,
    class_weight=class_weights,
    callbacks=[early_stopping],
    verbose=1
)

pred5, acc5, prec5, rec5, f15, cm5 = evaluate_model(
    model5, X_test_scaled, y_test
)

print("\nModel 5 Results")
print("Accuracy :", round(acc5, 4))
print("Precision:", round(prec5, 4))
print("Recall   :", round(rec5, 4))
print("F1 Score :", round(f15, 4))


NameError: name 'X_train_scaled' is not defined

In [18]:
# Model 5 Loss Graph

plt.figure(figsize=(8,5))
plt.plot(history5.history["loss"], label="Training Loss")
plt.plot(history5.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Binary Cross-Entropy Loss")
plt.title("Model 5 - Deep ReLU + Adam + Dropout")
plt.legend()
plt.show()


NameError: name 'history5' is not defined

<Figure size 800x500 with 0 Axes>

In [19]:
# 13. Model Comparison Table

results = pd.DataFrame({
    "Model": ["Model 1", "Model 2", "Model 3", "Model 4", "Model 5"],
    "Architecture": [
        "64 -> 32",
        "64 -> 32",
        "128 -> 64 -> 32",
        "64 -> 32",
        "128 -> 64 -> 32 -> 16"
    ],
    "Activation": ["ReLU", "Tanh", "ReLU", "Sigmoid", "ReLU"],
    "Optimizer": ["Adam", "SGD", "RMSprop", "Adam", "Adam"],
    "Epochs": [20, 50, 100, 50, 100],
    "Batch Size": [256, 256, 256, 256, 128],
    "Accuracy": [acc1, acc2, acc3, acc4, acc5],
    "Precision": [prec1, prec2, prec3, prec4, prec5],
    "Recall": [rec1, rec2, rec3, rec4, rec5],
    "F1": [f11, f12, f13, f14, f15]
})

display(results.round(4))


NameError: name 'acc1' is not defined

In [20]:
# 14. Select Best Model

# For fraud detection, F1-score is used as the primary selection metric,
# while Recall is also carefully considered.

best_index = results["F1"].idxmax()
best_model_name = results.loc[best_index, "Model"]

models = {
    "Model 1": model1,
    "Model 2": model2,
    "Model 3": model3,
    "Model 4": model4,
    "Model 5": model5
}

predictions = {
    "Model 1": pred1,
    "Model 2": pred2,
    "Model 3": pred3,
    "Model 4": pred4,
    "Model 5": pred5
}

confusion_matrices = {
    "Model 1": cm1,
    "Model 2": cm2,
    "Model 3": cm3,
    "Model 4": cm4,
    "Model 5": cm5
}

best_model = models[best_model_name]
best_predictions = predictions[best_model_name]
best_cm = confusion_matrices[best_model_name]

print("Best Performing Model:", best_model_name)
print("\nSelected primarily by highest F1-score, with Recall considered important for fraud detection.")
display(results.loc[[best_index]].round(4))


NameError: name 'results' is not defined

In [21]:
# 15. Classification Report for Best Model

print(classification_report(
    y_test,
    best_predictions,
    target_names=["Legitimate (0)", "Fraudulent (1)"],
    digits=4,
    zero_division=0
))


NameError: name 'y_test' is not defined

In [22]:
# 16. Confusion Matrix for Best Model

print("Confusion Matrix:")
print(best_cm)

plt.figure(figsize=(6,5))
plt.imshow(best_cm)
plt.title(f"Confusion Matrix - {best_model_name}")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.xticks([0, 1], ["Legitimate (0)", "Fraudulent (1)"])
plt.yticks([0, 1], ["Legitimate (0)", "Fraudulent (1)"])

for i in range(2):
    for j in range(2):
        plt.text(j, i, best_cm[i, j], ha="center", va="center")

plt.colorbar()
plt.show()


Confusion Matrix:


NameError: name 'best_cm' is not defined

In [23]:
# 17. Sample Predictions

sample_predictions = pd.DataFrame({
    "Actual Class": y_test.values[:20],
    "Predicted Class": best_predictions[:20]
})

sample_predictions["Actual Meaning"] = sample_predictions["Actual Class"].map({
    0: "Legitimate",
    1: "Fraudulent"
})

sample_predictions["Predicted Meaning"] = sample_predictions["Predicted Class"].map({
    0: "Legitimate",
    1: "Fraudulent"
})

display(sample_predictions)


NameError: name 'y_test' is not defined

### Prediction Interpretation

- **0 = Legitimate:** The ANN predicts that the transaction is normal/non-fraudulent.
- **1 = Fraudulent:** The ANN predicts that the transaction is potentially fraudulent.

The confusion matrix helps identify false positives and false negatives. In fraud detection, **false negatives** are especially important because they represent fraudulent transactions that the model failed to detect.


In [24]:
# 18. Final Best Model Performance

best_row = results.loc[best_index]

print("FINAL BEST ANN MODEL")
print("=" * 35)
print("Model        :", best_row["Model"])
print("Architecture :", best_row["Architecture"])
print("Activation   :", best_row["Activation"])
print("Optimizer    :", best_row["Optimizer"])
print("Epochs       :", best_row["Epochs"])
print("Batch Size   :", best_row["Batch Size"])
print("Accuracy     :", round(best_row["Accuracy"], 4))
print("Precision    :", round(best_row["Precision"], 4))
print("Recall       :", round(best_row["Recall"], 4))
print("F1 Score     :", round(best_row["F1"], 4))


NameError: name 'results' is not defined